### Intelligent RAG-Based Agent: End-to-End Guide
### Using FAISS + Free/Open-Source LLMs 

---

### Table of Contents
1. [Theory: What is RAG?](#theory)
2. [Architecture Overview](#architecture)
3. [Environment Setup](#setup)
4. [Document Loading & Chunking](#chunking)
5. [Embeddings & FAISS Vector Store](#embeddings)
6. [Retrieval Engine](#retrieval)
7. [Free LLM Integration (Multiple Options)](#llm)
8. [Building the RAG Agent](#agent)
9. [Advanced: Agentic RAG with Tools](#advanced)
10. [Evaluation & Testing](#evaluation)
11. [Production Tips](#production)

---

## 1.Theory: What is RAG? <a id='theory'></a>

### 1.1 The Core Problem

Large Language Models (LLMs) have two critical limitations:
- **Knowledge Cutoff**: They only know what was in their training data
- **Hallucination**: They confidently generate plausible but incorrect information

### 1.2 What is RAG?

**Retrieval-Augmented Generation (RAG)** is a technique that enhances LLMs by:
1. **Retrieving** relevant documents from a knowledge base at query time
2. **Augmenting** the LLM prompt with those retrieved documents
3. **Generating** an answer grounded in the retrieved context

```
┌─────────────────────────────────────────────────────────────┐
│                    RAG PIPELINE                             │
│                                                             │
│  User Query                                                 │
│      │                                                      │
│      ▼                                                      │
│  [Query Encoder]  ──→  Query Embedding                      │
│                              │                              │
│                              ▼                              │
│  Knowledge Base ──→  [FAISS Vector Search]                  │
│  (Documents)               │                               │
│                    Top-K Relevant Chunks                    │
│                              │                              │
│                              ▼                              │
│  [Prompt Builder]  ←─  Retrieved Context                    │
│      │                                                      │
│      ▼                                                      │
│  [LLM Generator]  ──→  Final Answer                         │
└─────────────────────────────────────────────────────────────┘
```

### 1.3 Key Concepts

| Concept | Description |
|---------|-------------|
| **Embedding** | Dense vector representation of text (semantic meaning) |
| **Vector Store** | Database optimized for similarity search on vectors |
| **FAISS** | Facebook AI Similarity Search — fast, free, local vector index |
| **Chunk** | A piece of a document (split for better retrieval granularity) |
| **Top-K Retrieval** | Return K most semantically similar chunks to the query |
| **Cosine Similarity** | Measure of angle between two vectors (0=different, 1=identical) |

### 1.4 Why FAISS?

FAISS (Facebook AI Similarity Search) is ideal because:
- ✅ **Completely free and local** — no API costs
- ✅ **Extremely fast** — handles millions of vectors
- ✅ **Multiple index types** — exact search, approximate (IVF, HNSW)
- ✅ **In-memory or disk** — flexible deployment
- ✅ **Battle-tested** — used at massive scale at Meta

### 1.5 The Math Behind It

**Cosine Similarity:**
$$\text{sim}(q, d) = \frac{q \cdot d}{\|q\| \|d\|}$$

**L2 Distance (used by FAISS default):**
$$\text{dist}(q, d) = \sqrt{\sum_{i=1}^{n}(q_i - d_i)^2}$$

FAISS normalizes vectors for cosine similarity. Smaller L2 distance = more similar.

## 2. 🏗️ Architecture Overview <a id='architecture'></a>

```
INDEXING PHASE (run once):                QUERY PHASE (run per query):
─────────────────────────                ──────────────────────────────
  Raw Documents                            User Question
       │                                        │
       ▼                                        ▼
  Text Splitter                          Embed Question
  (chunks ~500 tokens)                   (same model)
       │                                        │
       ▼                                        ▼
  Embed Chunks                           FAISS Search
  (sentence-transformers)                (Top-K similar)
       │                                        │
       ▼                                        ▼
  FAISS Index                            Build Prompt
  (save to disk)               [Context] + [Question]
                                                │
                                                ▼
                                        Free LLM Answer
                                        (local or free API)
```

### Free LLM Options Available

| Option | Type | Speed | Quality | Notes |
|--------|------|-------|---------|-------|
| `sentence-transformers` | Embeddings only | ⚡⚡⚡ | ⭐⭐⭐⭐ | Best free embeddings |
| `ollama` (local) | Full LLM | ⚡⚡ | ⭐⭐⭐⭐ | Llama3, Mistral, etc. |
| `Hugging Face` (local) | Full LLM | ⚡ | ⭐⭐⭐ | GPT2, Flan-T5, etc. |
| `Groq API` (free tier) | Cloud LLM | ⚡⚡⚡ | ⭐⭐⭐⭐⭐ | Free, fast Llama3 |
| `Together AI` (free tier)| Cloud LLM | ⚡⚡⚡ | ⭐⭐⭐⭐⭐ | Free credits |
| `Cohere` (free tier) | Cloud LLM | ⚡⚡⚡ | ⭐⭐⭐⭐ | Free trial |


## 3.  Environment Setup <a id='setup'></a>

In [2]:
# Core imports
import os
import json
import time
import pickle
import numpy as np
import faiss
from pathlib import Path
from typing import List, Dict, Tuple, Optional

# Sentence Transformers for embeddings
from sentence_transformers import SentenceTransformer

# Text processing
import re
from dataclasses import dataclass, field

print("Imports successful!")
print(f"FAISS version: {faiss.__version__}")
print(f"NumPy version: {np.__version__}")

C:\Users\Suyashi144893\AppData\Local\anaconda3\envs\pytorch_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports successful!
FAISS version: 1.13.2
NumPy version: 2.2.6


## 4.Document Loading & Chunking <a id='chunking'></a>

### Theory: Why Chunking Matters

Documents are split into chunks because:
- LLMs have **context window limits** (can't fit entire documents)
- **Smaller chunks = more precise retrieval** (you find the exact paragraph, not the whole book)
- **Embedding quality degrades** on very long texts

**Chunking strategies:**
- **Fixed size**: Split every N characters (simple, fast)
- **Sentence boundary**: Split on sentences (better quality)
- **Recursive**: Try paragraph → sentence → word (LangChain default)
- **Semantic**: Group by meaning (best, slowest)

In [3]:
@dataclass
class Document:
    """Represents a document chunk with metadata."""
    content: str
    metadata: Dict = field(default_factory=dict)
    doc_id: str = ""
    
    def __repr__(self):
        return f"Document(id={self.doc_id}, chars={len(self.content)}, source={self.metadata.get('source', 'unknown')})"


class TextSplitter:
    """
    Recursive text splitter that tries to split on natural boundaries.
    
    Strategy: paragraph → sentence → word → character
    With overlap to preserve context across chunk boundaries.
    """
    
    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 50):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        # Separators ordered from most to least preferred
        self.separators = ["\n\n", "\n", ". ", "! ", "? ", ", ", " ", ""]
    
    def split_text(self, text: str) -> List[str]:
        """Split text into chunks with overlap."""
        # Clean the text
        text = re.sub(r'\s+', ' ', text).strip()
        
        if len(text) <= self.chunk_size:
            return [text]
        
        chunks = []
        # Find best separator
        separator = ""
        for sep in self.separators:
            if sep in text:
                separator = sep
                break
        
        # Split on separator
        if separator:
            splits = text.split(separator)
        else:
            splits = list(text)
        
        # Merge splits into chunks
        current_chunk = ""
        for split in splits:
            candidate = current_chunk + separator + split if current_chunk else split
            if len(candidate) <= self.chunk_size:
                current_chunk = candidate
            else:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                # Handle overlap: keep last N chars of previous chunk
                if self.chunk_overlap > 0 and chunks:
                    overlap_text = chunks[-1][-self.chunk_overlap:]
                    current_chunk = overlap_text + " " + split
                else:
                    current_chunk = split
        
        if current_chunk.strip():
            chunks.append(current_chunk.strip())
        
        # Filter out very short chunks
        return [c for c in chunks if len(c) > 20]
    
    def split_documents(self, documents: List[Document]) -> List[Document]:
        """Split a list of documents into chunks."""
        chunked_docs = []
        for doc in documents:
            chunks = self.split_text(doc.content)
            for i, chunk in enumerate(chunks):
                chunked_doc = Document(
                    content=chunk,
                    metadata={**doc.metadata, 'chunk_index': i, 'total_chunks': len(chunks)},
                    doc_id=f"{doc.doc_id}_chunk_{i}"
                )
                chunked_docs.append(chunked_doc)
        return chunked_docs


# ─── Demo: Create sample documents ───────────────────────────────────────────
sample_texts = [
    {
        "source": "ai_basics.txt",
        "content": """
        Artificial Intelligence (AI) is the simulation of human intelligence processes by machines.
        These processes include learning, reasoning, and self-correction. Machine learning is a 
        subset of AI that gives computers the ability to learn without being explicitly programmed.
        Deep learning is a further subset that uses neural networks with many layers.
        
        Natural Language Processing (NLP) is a branch of AI that deals with the interaction 
        between computers and human language. Large Language Models (LLMs) like GPT-4 are 
        trained on massive text datasets and can generate human-like text.
        
        Transformers are the architecture behind modern LLMs. Introduced in 2017, they use 
        attention mechanisms to process sequences of data. The self-attention mechanism allows 
        the model to weigh the importance of different words in a sentence.
        """
    },
    {
        "source": "python_guide.txt",
        "content": """
        Python is a high-level, interpreted programming language known for its simple syntax.
        It was created by Guido van Rossum and first released in 1991. Python supports multiple 
        programming paradigms including procedural, object-oriented, and functional programming.
        
        Python has a rich ecosystem of libraries. NumPy is used for numerical computing.
        Pandas is used for data manipulation. Matplotlib and Seaborn are used for visualization.
        Scikit-learn provides machine learning algorithms. TensorFlow and PyTorch are used for 
        deep learning. FastAPI is used for building APIs quickly.
        
        Virtual environments in Python allow you to create isolated environments for different 
        projects. You can create one using: python -m venv myenv. Activate it with source 
        myenv/bin/activate on Linux/Mac or myenv\\Scripts\\activate on Windows.
        """
    },
    {
        "source": "vector_db_guide.txt",
        "content": """
        Vector databases store data as high-dimensional vectors (embeddings). They enable 
        semantic similarity search, which is fundamentally different from keyword search.
        While keyword search looks for exact word matches, semantic search finds conceptually 
        similar content even with different wording.
        
        FAISS (Facebook AI Similarity Search) is an open-source library for efficient similarity 
        search. It supports billions of vectors and multiple index types. IndexFlatL2 does exact 
        brute-force search. IndexIVFFlat clusters vectors for faster approximate search.
        IndexHNSWFlat uses graph-based approximate search, ideal for production.
        
        Embeddings are created by encoding text with a model like sentence-transformers. 
        All-MiniLM-L6-v2 is a popular free model that creates 384-dimensional embeddings. 
        All-mpnet-base-v2 creates 768-dimensional embeddings with higher quality.
        """
    }
]

# Create Document objects
raw_docs = [
    Document(
        content=item['content'], 
        metadata={'source': item['source']},
        doc_id=f"doc_{i}"
    )
    for i, item in enumerate(sample_texts)
]

# Split into chunks
splitter = TextSplitter(chunk_size=400, chunk_overlap=50)
chunked_docs = splitter.split_documents(raw_docs)

print(f"📄 Original documents: {len(raw_docs)}")
print(f"🔪 After chunking: {len(chunked_docs)} chunks")
print("\nFirst 3 chunks:")
for doc in chunked_docs[:3]:
    print(f"  {doc}")
    print(f"  Content preview: {doc.content[:100]}...\n")

📄 Original documents: 3
🔪 After chunking: 9 chunks

First 3 chunks:
  Document(id=doc_0_chunk_0, chars=348, source=ai_basics.txt)
  Content preview: Artificial Intelligence (AI) is the simulation of human intelligence processes by machines. These pr...

  Document(id=doc_0_chunk_1, chars=335, source=ai_basics.txt)
  Content preview: subset that uses neural networks with many layers Natural Language Processing (NLP) is a branch of A...

  Document(id=doc_0_chunk_2, chars=234, source=ai_basics.txt)
  Content preview: ansformers are the architecture behind modern LLMs Introduced in 2017, they use attention mechanisms...



## 5.Embeddings & FAISS Vector Store <a id='embeddings'></a>

### Theory: How Embeddings Work

An **embedding model** maps text → dense vector:
- `"king"` → `[0.2, -0.5, 0.8, ...]` (384 numbers)
- `"queen"` → `[0.19, -0.48, 0.78, ...]` (very similar!)
- `"bicycle"` → `[0.9, 0.1, -0.3, ...]` (very different)

The famous word2vec property: `king - man + woman ≈ queen`

### FAISS Index Types

```
IndexFlatL2    → Exact search, brute force O(n). Best for < 10k vectors.
IndexIVFFlat   → Approximate. Clusters data, searches nearest clusters.
               → Best for 10k–1M vectors. Requires training.
IndexHNSWFlat  → Graph-based. Best speed/accuracy tradeoff.
               → Production standard for > 100k vectors.
IndexPQ        → Product Quantization. Compressed. Best for huge datasets.
```

In [4]:
class FAISSVectorStore:
    """
    A complete vector store using FAISS for similarity search.
    
    Handles:
    - Loading embedding model
    - Encoding documents to vectors
    - Building and managing FAISS index
    - Similarity search
    - Save/load to disk
    """
    
    def __init__(
        self, 
        model_name: str = "all-MiniLM-L6-v2",
        index_type: str = "flat"  # 'flat', 'ivf', 'hnsw'
    ):
        print(f"🔄 Loading embedding model: {model_name}...")
        self.model = SentenceTransformer(model_name)
        self.dimension = self.model.get_sentence_embedding_dimension()
        self.index_type = index_type
        self.index = None
        self.documents = []  # Store original documents
        self.embeddings = None
        print(f"✅ Model loaded! Embedding dimension: {self.dimension}")
    
    def _build_index(self, n_docs: int) -> faiss.Index:
        """Build the appropriate FAISS index."""
        if self.index_type == "flat":
            # Exact search — best for small datasets
            index = faiss.IndexFlatL2(self.dimension)
            
        elif self.index_type == "ivf":
            # Inverted File Index — good for medium datasets
            n_clusters = max(4, int(np.sqrt(n_docs)))
            quantizer = faiss.IndexFlatL2(self.dimension)
            index = faiss.IndexIVFFlat(quantizer, self.dimension, n_clusters)
            
        elif self.index_type == "hnsw":
            # Hierarchical Navigable Small World — best for production
            index = faiss.IndexHNSWFlat(self.dimension, 32)  # 32 = M connections
            index.hnsw.efConstruction = 200
            index.hnsw.efSearch = 50
        
        else:
            raise ValueError(f"Unknown index type: {self.index_type}")
        
        # Wrap with IDMap to support add_with_ids
        if self.index_type != "hnsw":
            index = faiss.IndexIDMap(index)
        
        return index
    
    def embed_texts(self, texts: List[str], batch_size: int = 32) -> np.ndarray:
        """Convert texts to normalized embeddings."""
        print(f"⚙️  Encoding {len(texts)} texts...")
        embeddings = self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=len(texts) > 50,
            normalize_embeddings=True  # For cosine similarity via L2
        )
        return embeddings.astype('float32')
    
    def add_documents(self, documents: List[Document]):
        """Add documents to the vector store."""
        if not documents:
            raise ValueError("No documents provided")
        
        # Extract texts
        texts = [doc.content for doc in documents]
        
        # Create embeddings
        embeddings = self.embed_texts(texts)
        
        # Build index
        self.index = self._build_index(len(documents))
        
        # Train if needed (IVF requires training)
        if self.index_type == "ivf":
            print("🏋️  Training IVF index...")
            # For IVFFlat wrapped in IDMap, we need to train the inner index
            inner = faiss.downcast_index(self.index.index)
            inner.train(embeddings)
        
        # Add vectors with IDs
        ids = np.arange(len(documents)).astype('int64')
        
        if hasattr(self.index, 'add_with_ids'):
            self.index.add_with_ids(embeddings, ids)
        else:
            self.index.add(embeddings)
        
        # Store documents and embeddings
        self.documents = documents
        self.embeddings = embeddings
        
        print(f"✅ Added {len(documents)} documents to FAISS index")
        print(f"   Index contains {self.index.ntotal} vectors")
    
    def similarity_search(
        self, 
        query: str, 
        k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Tuple[Document, float]]:
        """
        Search for most similar documents.
        
        Returns list of (document, similarity_score) tuples.
        Score is 1 - L2_distance (higher = more similar).
        """
        if self.index is None:
            raise ValueError("Index is empty. Call add_documents() first.")
        
        # Embed query
        query_embedding = self.embed_texts([query])
        
        # Search
        distances, indices = self.index.search(query_embedding, k)
        
        # Convert L2 distances to similarity scores (0 to 1)
        # Since embeddings are normalized, L2^2 = 2 - 2*cosine_sim
        # So cosine_sim = 1 - L2^2/2
        results = []
        for dist, idx in zip(distances[0], indices[0]):
            if idx == -1:  # FAISS returns -1 for not found
                continue
            similarity = float(1 - dist / 2)  # Convert to cosine similarity
            if similarity >= score_threshold:
                results.append((self.documents[idx], similarity))
        
        return results
    
    def save(self, directory: str):
        """Save index and documents to disk."""
        os.makedirs(directory, exist_ok=True)
        faiss.write_index(self.index, os.path.join(directory, "index.faiss"))
        with open(os.path.join(directory, "documents.pkl"), 'wb') as f:
            pickle.dump(self.documents, f)
        print(f"💾 Saved to {directory}")
    
    def load(self, directory: str):
        """Load index and documents from disk."""
        self.index = faiss.read_index(os.path.join(directory, "index.faiss"))
        with open(os.path.join(directory, "documents.pkl"), 'rb') as f:
            self.documents = pickle.load(f)
        print(f"📂 Loaded {len(self.documents)} documents from {directory}")


# ─── Build the vector store ───────────────────────────────────────────────────
vector_store = FAISSVectorStore(
    model_name="all-MiniLM-L6-v2",  # Free, fast, good quality
    index_type="flat"                 # Exact search for small datasets
)

vector_store.add_documents(chunked_docs)

# Save for later
vector_store.save("./rag_index")

🔄 Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 5031.78it/s]
C:\Users\Suyashi144893\AppData\Local\Temp\1\ipykernel_26060\2396942597.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dimension = self.model.get_sentence_embedding_dimension()


✅ Model loaded! Embedding dimension: 384
⚙️  Encoding 9 texts...
✅ Added 9 documents to FAISS index
   Index contains 9 vectors
💾 Saved to ./rag_index


In [5]:
# ─── Test the retrieval ───────────────────────────────────────────────────────
test_query = "How do neural networks learn?"

print(f" Query: '{test_query}'")
print("-" * 60)

results = vector_store.similarity_search(test_query, k=3)

for i, (doc, score) in enumerate(results):
    print(f"\n Result {i+1} (similarity: {score:.3f})")
    print(f"   Source: {doc.metadata.get('source', 'unknown')}")
    print(f"   Content: {doc.content[:200]}...")

🔍 Query: 'How do neural networks learn?'
------------------------------------------------------------
⚙️  Encoding 1 texts...

 Result 1 (similarity: 0.469)
   Source: ai_basics.txt
   Content: Artificial Intelligence (AI) is the simulation of human intelligence processes by machines. These processes include learning, reasoning, and self-correction. Machine learning is a subset of AI that gi...

 Result 2 (similarity: 0.426)
   Source: ai_basics.txt
   Content: subset that uses neural networks with many layers Natural Language Processing (NLP) is a branch of AI that deals with the interaction between computers and human language. Large Language Models (LLMs)...

 Result 3 (similarity: 0.305)
   Source: ai_basics.txt
   Content: ansformers are the architecture behind modern LLMs Introduced in 2017, they use attention mechanisms to process sequences of data. The self-attention mechanism allows the model to weigh the importance...


## 6.Retrieval Engine <a id='retrieval'></a>

### Theory: Advanced Retrieval Strategies

Basic retrieval can be improved with:

1. **MMR (Maximal Marginal Relevance)**: Balance relevance + diversity
   - Avoids retrieving 5 nearly-identical chunks
   - Formula: `MMR = λ * Relevance - (1-λ) * MaxSimilarityToAlreadySelected`

2. **Re-ranking**: Use a cross-encoder to re-score retrieved docs
   - Two-stage: fast retrieval → accurate re-scoring

3. **Hybrid Search**: Combine dense (semantic) + sparse (BM25/keyword)
   - Handles both semantic and lexical queries well

In [6]:
class RetrievalEngine:
    """
    Advanced retrieval with MMR and filtering.
    """
    
    def __init__(self, vector_store: FAISSVectorStore):
        self.vector_store = vector_store
    
    def retrieve(self, query: str, k: int = 4, method: str = "similarity") -> List[Document]:
        """Main retrieval interface."""
        if method == "similarity":
            return self._similarity_retrieve(query, k)
        elif method == "mmr":
            return self._mmr_retrieve(query, k)
        else:
            raise ValueError(f"Unknown method: {method}")
    
    def _similarity_retrieve(self, query: str, k: int) -> List[Document]:
        """Simple top-k similarity retrieval."""
        results = self.vector_store.similarity_search(query, k=k)
        return [doc for doc, _ in results]
    
    def _mmr_retrieve(self, query: str, k: int, lambda_val: float = 0.5) -> List[Document]:
        """
        Maximal Marginal Relevance retrieval.
        
        Balances relevance to query vs diversity among results.
        lambda_val: 1.0 = pure relevance, 0.0 = pure diversity
        """
        # Fetch more candidates than needed
        candidates = self.vector_store.similarity_search(query, k=k*3)
        if not candidates:
            return []
        
        # Get query embedding
        query_emb = self.vector_store.embed_texts([query])[0]
        
        # Get candidate embeddings
        candidate_docs = [doc for doc, _ in candidates]
        candidate_scores = [score for _, score in candidates]
        
        if len(candidate_docs) == 0:
            return []
        
        candidate_embs = self.vector_store.embed_texts(
            [doc.content for doc in candidate_docs]
        )
        
        selected_indices = []
        remaining = list(range(len(candidate_docs)))
        
        # Select k documents using MMR
        for _ in range(min(k, len(candidates))):
            if not remaining:
                break
            
            if not selected_indices:
                # First pick: most relevant
                best = max(remaining, key=lambda i: candidate_scores[i])
            else:
                # MMR: balance relevance and diversity
                mmr_scores = {}
                selected_embs = candidate_embs[selected_indices]
                
                for i in remaining:
                    relevance = candidate_scores[i]
                    # Max similarity to already selected
                    sims_to_selected = np.dot(selected_embs, candidate_embs[i])
                    max_redundancy = float(np.max(sims_to_selected))
                    mmr_scores[i] = lambda_val * relevance - (1 - lambda_val) * max_redundancy
                
                best = max(remaining, key=lambda i: mmr_scores[i])
            
            selected_indices.append(best)
            remaining.remove(best)
        
        return [candidate_docs[i] for i in selected_indices]
    
    def format_context(self, documents: List[Document]) -> str:
        """Format retrieved documents as context string."""
        context_parts = []
        for i, doc in enumerate(documents, 1):
            source = doc.metadata.get('source', 'unknown')
            context_parts.append(
                f"[Source {i}: {source}]\n{doc.content}"
            )
        return "\n\n---\n\n".join(context_parts)


# ─── Test retrieval engine ────────────────────────────────────────────────────
retriever = RetrievalEngine(vector_store)

queries = [
    "What is FAISS and how does it work?",
    "How do I create Python virtual environments?",
    "What are transformers in AI?"
]

for q in queries:
    print(f"\n Query: {q}")
    docs = retriever.retrieve(q, k=2, method="mmr")
    for j, doc in enumerate(docs):
        print(f"  [{j+1}] {doc.metadata.get('source')} | {doc.content[:120]}...")


🔍 Query: What is FAISS and how does it work?
⚙️  Encoding 1 texts...
⚙️  Encoding 1 texts...
⚙️  Encoding 6 texts...
  [1] vector_db_guide.txt | Vector databases store data as high-dimensional vectors (embeddings). They enable semantic similarity search, which is f...
  [2] python_guide.txt | Python is a high-level, interpreted programming language known for its simple syntax. It was created by Guido van Rossum...

🔍 Query: How do I create Python virtual environments?
⚙️  Encoding 1 texts...
⚙️  Encoding 1 texts...
⚙️  Encoding 6 texts...
  [1] python_guide.txt | Python is a high-level, interpreted programming language known for its simple syntax. It was created by Guido van Rossum...
  [2] python_guide.txt | ts. You can create one using: python -m venv myenv Activate it with source myenv/bin/activate on Linux/Mac or myenv\Scri...

🔍 Query: What are transformers in AI?
⚙️  Encoding 1 texts...
⚙️  Encoding 1 texts...
⚙️  Encoding 6 texts...
  [1] ai_basics.txt | subset that uses neural

## 7.Free LLM Integration <a id='llm'></a>

We provide **4 options** — use whichever fits your setup:

| Option | Requirements | Quality |
|--------|-------------|------|
| Option A | `transformers` only (fully offline) |
| Option B | Ollama installed locally |
| Option C | Free Groq API key | 
| Option D | Hugging Face Inference API | 

In [7]:
import requests
from abc import ABC, abstractmethod

class BaseLLM(ABC):
    """Abstract base class for LLM backends."""
    
    @abstractmethod
    def generate(self, prompt: str, max_tokens: int = 512) -> str:
        pass
    
    def __call__(self, prompt: str, **kwargs) -> str:
        return self.generate(prompt, **kwargs)


# ─────────────────────────────────────────────────────────────────────────────
# OPTION A: Local Hugging Face model (fully offline, no API key needed)
# ─────────────────────────────────────────────────────────────────────────────
class HuggingFaceLocalLLM(BaseLLM):
    """
    Uses a local Hugging Face model.
    
    Good free models:
    - 'google/flan-t5-base'        (250MB, instruction-tuned, good for Q&A)
    - 'google/flan-t5-large'       (780MB, better quality)
    - 'microsoft/phi-2'            (2.7B, surprisingly capable)
    - 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'  (1.1B, chat-tuned)
    """
    
    def __init__(self, model_name: str = "google/flan-t5-base"):
        from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
        print(f"🔄 Loading {model_name}...")
        
        if "t5" in model_name.lower():
            self.pipe = pipeline(
                "text2text-generation",
                model=model_name,
                max_new_tokens=256
            )
        else:
            self.pipe = pipeline(
                "text-generation",
                model=model_name,
                max_new_tokens=256,
                do_sample=True,
                temperature=0.7
            )
        
        self.is_seq2seq = "t5" in model_name.lower()
        print(f"{model_name} loaded!")
    
    def generate(self, prompt: str, max_tokens: int = 256) -> str:
        result = self.pipe(prompt, max_new_tokens=max_tokens)
        if self.is_seq2seq:
            return result[0]['generated_text']
        else:
            text = result[0]['generated_text']
            # Remove the input prompt from output
            return text[len(prompt):].strip()


# ─────────────────────────────────────────────────────────────────────────────
# OPTION B: Ollama (local, requires Ollama installation)
# ─────────────────────────────────────────────────────────────────────────────
class OllamaLLM(BaseLLM):
    """
    Uses locally running Ollama.
    Install: https://ollama.ai
    Pull model: ollama pull llama3.2 (or mistral, gemma2, phi3, etc.)
    """
    
    def __init__(self, model: str = "llama3.2", base_url: str = "http://localhost:11434"):
        self.model = model
        self.base_url = base_url
        # Test connection
        try:
            resp = requests.get(f"{base_url}/api/tags", timeout=3)
            models = [m['name'] for m in resp.json().get('models', [])]
            print(f"✅ Ollama connected! Available models: {models}")
        except:
            print("⚠️  Ollama not running. Start with: ollama serve")
    
    def generate(self, prompt: str, max_tokens: int = 512) -> str:
        response = requests.post(
            f"{self.base_url}/api/generate",
            json={
                "model": self.model,
                "prompt": prompt,
                "stream": False,
                "options": {"num_predict": max_tokens, "temperature": 0.1}
            },
            timeout=120
        )
        return response.json()["response"].strip()


# ─────────────────────────────────────────────────────────────────────────────
# OPTION C: Groq API (FREE tier — fastest inference, Llama3)
# ─────────────────────────────────────────────────────────────────────────────
class GroqLLM(BaseLLM):
    """
    Groq API — free tier available at console.groq.com
    Models: llama-3.3-70b-versatile, llama-3.1-8b-instant, mixtral-8x7b-32768
    """
    
    def __init__(self, api_key: str, model: str = "llama-3.1-8b-instant"):
        self.api_key = api_key
        self.model = model
        self.base_url = "https://api.groq.com/openai/v1/chat/completions"
        print(f"✅ Groq configured with model: {model}")
    
    def generate(self, prompt: str, max_tokens: int = 512) -> str:
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        data = {
            "model": self.model,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens,
            "temperature": 0.1
        }
        response = requests.post(self.base_url, headers=headers, json=data)
        return response.json()["choices"][0]["message"]["content"].strip()


# ─────────────────────────────────────────────────────────────────────────────
# OPTION D: HuggingFace Inference API (free tier)
# ─────────────────────────────────────────────────────────────────────────────
class HuggingFaceAPILLM(BaseLLM):
    """
    HuggingFace Inference API — free tier at huggingface.co
    Get token: https://huggingface.co/settings/tokens
    """
    
    def __init__(self, hf_token: str, model: str = "mistralai/Mistral-7B-Instruct-v0.3"):
        self.hf_token = hf_token
        self.model = model
        self.url = f"https://api-inference.huggingface.co/models/{model}"
        print(f"✅ HuggingFace API configured: {model}")
    
    def generate(self, prompt: str, max_tokens: int = 512) -> str:
        headers = {"Authorization": f"Bearer {self.hf_token}"}
        payload = {
            "inputs": prompt,
            "parameters": {"max_new_tokens": max_tokens, "temperature": 0.1}
        }
        response = requests.post(self.url, headers=headers, json=payload)
        result = response.json()
        if isinstance(result, list):
            return result[0].get('generated_text', '').replace(prompt, '').strip()
        return str(result)


# ─────────────────────────────────────────────────────────────────────────────
# MOCK LLM for demonstration (no installation needed)
# ─────────────────────────────────────────────────────────────────────────────
class MockLLM(BaseLLM):
    """Mock LLM for testing RAG pipeline without a real model."""
    
    def generate(self, prompt: str, max_tokens: int = 512) -> str:
        # Extract context from the prompt
        if "Context:" in prompt:
            context_start = prompt.find("Context:") + 8
            context_end = prompt.find("\n\nQuestion:")
            context = prompt[context_start:context_end].strip()[:300]
            question = prompt.split("Question:")[-1].strip()
            return (
                f"[MOCK ANSWER] Based on the provided context, I found relevant information.\n"
                f"Question asked: '{question}'\n"
                f"Context excerpt: '{context[:150]}...'\n"
                f"(Replace MockLLM with a real model for actual answers)"
            )
        return "[MOCK] No context provided."


# ─── Choose your LLM ─────────────────────────────────────────────────────────
print("Available LLM options:")
print("  A) HuggingFaceLocalLLM('google/flan-t5-base')  — fully offline")
print("  B) OllamaLLM('llama3.2')                       — requires Ollama")
print("  C) GroqLLM(api_key='YOUR_KEY')                 — free API key")
print("  D) HuggingFaceAPILLM(hf_token='YOUR_TOKEN')   — free HF token")
print("  *) MockLLM()                                   — for testing")
print()

# Using MockLLM for this demo — swap with your preferred option!
# llm = HuggingFaceLocalLLM('google/flan-t5-base')  # Uncomment to use
# llm = OllamaLLM('llama3.2')                        # Uncomment to use
# llm = GroqLLM(api_key='YOUR_GROQ_KEY')             # Uncomment to use
llm = MockLLM()
print("✅ MockLLM initialized (swap for real model in production)")

Available LLM options:
  A) HuggingFaceLocalLLM('google/flan-t5-base')  — fully offline
  B) OllamaLLM('llama3.2')                       — requires Ollama
  C) GroqLLM(api_key='YOUR_KEY')                 — free API key
  D) HuggingFaceAPILLM(hf_token='YOUR_TOKEN')   — free HF token
  *) MockLLM()                                   — for testing

✅ MockLLM initialized (swap for real model in production)


## 8.Building the RAG Agent <a id='agent'></a>

### Theory: Prompt Engineering for RAG

The prompt is the bridge between retrieved context and LLM generation.

**Key elements of a good RAG prompt:**
1. **System instruction** — Define the LLM's role and behavior
2. **Context section** — Retrieved documents clearly labeled
3. **User question** — The actual query
4. **Grounding instruction** — "Only use the provided context"
5. **Fallback instruction** — "If not in context, say so"

In [8]:
class RAGAgent:
    """
    A complete RAG Agent that:
    1. Accepts a user question
    2. Retrieves relevant documents
    3. Builds an augmented prompt
    4. Generates a grounded answer
    5. Returns answer with sources
    """
    
    def __init__(
        self,
        vector_store: FAISSVectorStore,
        llm: BaseLLM,
        k: int = 4,
        retrieval_method: str = "mmr"
    ):
        self.retriever = RetrievalEngine(vector_store)
        self.llm = llm
        self.k = k
        self.retrieval_method = retrieval_method
        self.chat_history = []  # For multi-turn conversation
    
    def _build_prompt(self, question: str, context: str, history: List = None) -> str:
        """
        Build the RAG prompt.
        
        Structure:
        [System Role]
        [Retrieved Context]
        [Conversation History (optional)]
        [User Question]
        [Answer instruction]
        """
        prompt = """You are a helpful AI assistant. Answer questions based ONLY on the provided context.
If the context doesn't contain enough information to answer the question, say so clearly.
Do not make up information. Be concise and accurate.

Context:
{context}
""".format(context=context)
        
        # Add conversation history for multi-turn
        if history and len(history) > 0:
            prompt += "\nConversation History:\n"
            for turn in history[-3:]:  # Last 3 turns
                prompt += f"Human: {turn['question']}\nAssistant: {turn['answer']}\n"
        
        prompt += f"\nQuestion: {question}\n\nAnswer:"
        return prompt
    
    def query(
        self, 
        question: str,
        verbose: bool = True,
        remember: bool = True
    ) -> Dict:
        """
        Process a question through the RAG pipeline.
        
        Returns:
            dict with 'answer', 'sources', 'retrieved_docs', 'latency'
        """
        start_time = time.time()
        
        if verbose:
            print(f"\n{'='*60}")
            print(f"💬 Question: {question}")
            print('='*60)
        
        # Step 1: Retrieve relevant documents
        if verbose: print("\n📥 Step 1: Retrieving relevant documents...")
        retrieved_docs = self.retriever.retrieve(
            question, k=self.k, method=self.retrieval_method
        )
        
        if verbose:
            for i, doc in enumerate(retrieved_docs):
                print(f"   [{i+1}] {doc.metadata.get('source')} — {doc.content[:80]}...")
        
        # Step 2: Format context
        if verbose: print("\n📝 Step 2: Building context...")
        context = self.retriever.format_context(retrieved_docs)
        
        # Step 3: Build prompt
        if verbose: print("\n🔧 Step 3: Building prompt...")
        prompt = self._build_prompt(
            question, context, 
            history=self.chat_history if remember else None
        )
        
        if verbose:
            print(f"   Prompt length: {len(prompt)} chars")
        
        # Step 4: Generate answer
        if verbose: print("\n🤖 Step 4: Generating answer...")
        answer = self.llm.generate(prompt)
        
        # Step 5: Extract sources
        sources = list(set(
            doc.metadata.get('source', 'unknown') 
            for doc in retrieved_docs
        ))
        
        latency = time.time() - start_time
        
        # Save to history
        if remember:
            self.chat_history.append({
                "question": question,
                "answer": answer
            })
        
        result = {
            "question": question,
            "answer": answer,
            "sources": sources,
            "retrieved_docs": retrieved_docs,
            "latency_seconds": round(latency, 2)
        }
        
        if verbose:
            print(f"\n{'─'*60}")
            print(f"✅ Answer:\n{answer}")
            print(f"\n📚 Sources: {sources}")
            print(f"⏱️  Latency: {latency:.2f}s")
        
        return result
    
    def clear_history(self):
        """Clear conversation history."""
        self.chat_history = []
        print("🗑️  Chat history cleared.")


# ─── Initialize and test the RAG Agent ───────────────────────────────────────
agent = RAGAgent(
    vector_store=vector_store,
    llm=llm,
    k=3,
    retrieval_method="mmr"
)

# Test query
result = agent.query("What is FAISS and what are its index types?")


💬 Question: What is FAISS and what are its index types?

📥 Step 1: Retrieving relevant documents...
⚙️  Encoding 1 texts...
⚙️  Encoding 1 texts...
⚙️  Encoding 9 texts...
   [1] vector_db_guide.txt — Vector databases store data as high-dimensional vectors (embeddings). They enabl...
   [2] python_guide.txt — Python is a high-level, interpreted programming language known for its simple sy...
   [3] ai_basics.txt — ansformers are the architecture behind modern LLMs Introduced in 2017, they use ...

📝 Step 2: Building context...

🔧 Step 3: Building prompt...
   Prompt length: 1407 chars

🤖 Step 4: Generating answer...

────────────────────────────────────────────────────────────
✅ Answer:
[MOCK ANSWER] Based on the provided context, I found relevant information.
Question asked: 'What is FAISS and what are its index types?

Answer:'
Context excerpt: '[Source 1: vector_db_guide.txt]
Vector databases store data as high-dimensional vectors (embeddings). They enable semantic similarity searc

In [9]:
# ─── Multi-turn conversation demo ────────────────────────────────────────────
print("🗨️  Multi-turn Conversation Demo")
print("="*60)

agent.clear_history()

questions = [
    "What is Python used for?",
    "What libraries does it have for machine learning?",  # Refers back to Python
    "How do I set up a virtual environment?"
]

for q in questions:
    result = agent.query(q, verbose=True)
    print()

🗨️  Multi-turn Conversation Demo
🗑️  Chat history cleared.

💬 Question: What is Python used for?

📥 Step 1: Retrieving relevant documents...
⚙️  Encoding 1 texts...
⚙️  Encoding 1 texts...
⚙️  Encoding 9 texts...
   [1] python_guide.txt — Python is a high-level, interpreted programming language known for its simple sy...
   [2] ai_basics.txt — ansformers are the architecture behind modern LLMs Introduced in 2017, they use ...
   [3] ai_basics.txt — Artificial Intelligence (AI) is the simulation of human intelligence processes b...

📝 Step 2: Building context...

🔧 Step 3: Building prompt...
   Prompt length: 1340 chars

🤖 Step 4: Generating answer...

────────────────────────────────────────────────────────────
✅ Answer:
[MOCK ANSWER] Based on the provided context, I found relevant information.
Question asked: 'What is Python used for?

Answer:'
Context excerpt: '[Source 1: python_guide.txt]
Python is a high-level, interpreted programming language known for its simple syntax. It was cr

## 9. Advanced: Agentic RAG with Tools <a id='advanced'></a>

### Theory: What Makes a RAG *Agent*?

A basic RAG system retrieves once and answers. An **agentic RAG** can:
- **Decide** whether retrieval is even needed
- **Refine queries** if first retrieval isn't good enough
- **Use tools** (calculator, web search, code execution)
- **Reason step-by-step** before answering
- **Route** questions to different knowledge bases

```
User Question
     │
     ▼
[Intent Router] ──→ Is retrieval needed?
     │                     │
     │         Yes ←───────┘
     │
     ▼
[Query Rewriter] ──→ Optimize for retrieval
     │
     ▼
[FAISS Retrieve] ──→ Get top-K chunks
     │
     ▼
[Relevance Check] ──→ Good enough?
     │                     │
     │         No ──→ [Retry with new query]
     │
     ▼
[LLM Generate] ──→ Answer
     │
     ▼
[Hallucination Check] ──→ Is answer supported?
     │
     ▼
Final Answer
```

In [10]:
class AgenticRAG:
    """
    An advanced RAG agent with:
    - Query decomposition for complex questions
    - Query rewriting for better retrieval
    - Relevance grading
    - Answer grounding check
    - Tool use (calculator, date, code)
    """
    
    def __init__(self, vector_store: FAISSVectorStore, llm: BaseLLM):
        self.retriever = RetrievalEngine(vector_store)
        self.llm = llm
        self.tools = self._register_tools()
        self.conversation = []
    
    # ── Tool Registration ─────────────────────────────────────────────────────
    def _register_tools(self) -> Dict:
        """Register available tools."""
        return {
            "calculator": self._tool_calculator,
            "get_date": self._tool_get_date,
            "word_count": self._tool_word_count,
            "summarize": self._tool_summarize,
        }
    
    def _tool_calculator(self, expression: str) -> str:
        """Safely evaluate math expressions."""
        try:
            # Safe eval for math only
            allowed = set('0123456789+-*/().% ')
            if all(c in allowed for c in expression):
                result = eval(expression)
                return f"Calculator result: {result}"
            return "Invalid expression"
        except Exception as e:
            return f"Error: {e}"
    
    def _tool_get_date(self, _: str = "") -> str:
        """Return current date."""
        from datetime import datetime
        return f"Current date: {datetime.now().strftime('%Y-%m-%d %H:%M')}"
    
    def _tool_word_count(self, text: str) -> str:
        """Count words in text."""
        return f"Word count: {len(text.split())}"
    
    def _tool_summarize(self, text: str) -> str:
        """Summarize text."""
        prompt = f"Summarize in 2 sentences:\n{text[:1000]}"
        return self.llm.generate(prompt, max_tokens=100)
    
    # ── Query Rewriting ───────────────────────────────────────────────────────
    def rewrite_query(self, question: str, history: List = None) -> str:
        """
        Rewrite query for better retrieval.
        Handles:
        - Resolving pronouns using history
        - Expanding abbreviations
        - Making implicit context explicit
        """
        if not history or len(history) == 0:
            return question  # Nothing to resolve
        
        context = "; ".join([f"Q: {h['question']}" for h in history[-2:]])
        
        prompt = f"""Given this conversation context: [{context}]
Rewrite this question to be self-contained (resolve pronouns like 'it', 'they', 'that'):
Original: {question}
Rewritten (just the question, no explanation):"""
        
        rewritten = self.llm.generate(prompt, max_tokens=100)
        # Clean up the output
        rewritten = rewritten.strip().strip('"').strip("'")
        return rewritten if rewritten else question
    
    # ── Relevance Grading ─────────────────────────────────────────────────────
    def grade_relevance(self, question: str, context: str) -> float:
        """
        Grade how relevant the retrieved context is.
        Returns score 0.0 to 1.0.
        """
        # Keyword overlap as simple proxy (replace with LLM grading in production)
        question_words = set(question.lower().split())
        context_words = set(context.lower().split())
        # Remove stopwords
        stopwords = {'a', 'an', 'the', 'is', 'are', 'was', 'were', 'be', 'been',
                    'being', 'have', 'has', 'had', 'do', 'does', 'did', 'will',
                    'would', 'could', 'should', 'may', 'might', 'can', 'what',
                    'how', 'why', 'when', 'where', 'which', 'who', 'that', 'this'}
        q_words = question_words - stopwords
        if not q_words:
            return 0.5
        overlap = len(q_words.intersection(context_words))
        return min(1.0, overlap / len(q_words))
    
    # ── Tool Detection ────────────────────────────────────────────────────────
    def detect_tool_need(self, question: str) -> Optional[str]:
        """Detect if a tool should be used."""
        q = question.lower()
        if any(op in q for op in ['calculate', 'compute', 'what is', 'how much']) and \
           any(c in question for c in '0123456789'):
            return "calculator"
        if any(word in q for word in ['today', 'current date', 'what date', 'what time']):
            return "get_date"
        return None
    
    # ── Main Query Method ─────────────────────────────────────────────────────
    def query(self, question: str, verbose: bool = True) -> Dict:
        """Agentic query with reasoning steps."""
        if verbose:
            print(f"\n{'='*65}")
            print(f"🧠 AGENTIC RAG")
            print(f"💬 Question: {question}")
            print('='*65)
        
        steps = []
        
        # Step 0: Check for tool use
        tool_name = self.detect_tool_need(question)
        if tool_name:
            if verbose: print(f"\n🔧 Detected tool: {tool_name}")
            tool_result = self.tools[tool_name](question)
            steps.append(f"Tool used: {tool_name} → {tool_result}")
        
        # Step 1: Rewrite query
        if verbose: print("\n📝 Step 1: Query analysis...")
        rewritten_q = self.rewrite_query(question, self.conversation)
        if rewritten_q != question:
            if verbose: print(f"   Rewritten: '{rewritten_q}'")
        else:
            if verbose: print(f"   No rewriting needed")
        steps.append(f"Query: {rewritten_q}")
        
        # Step 2: Retrieve
        if verbose: print("\n📥 Step 2: Retrieval...")
        docs = self.retriever.retrieve(rewritten_q, k=4, method="mmr")
        context = self.retriever.format_context(docs)
        
        # Step 3: Grade relevance
        relevance_score = self.grade_relevance(question, context)
        if verbose: print(f"   Relevance score: {relevance_score:.2f}")
        
        # Step 4: If relevance is low, retry with broader query
        if relevance_score < 0.2:
            if verbose: print("   ⚠️  Low relevance — retrying with broader query")
            broader_q = " ".join(rewritten_q.split()[:5])  # Use first 5 words
            docs = self.retriever.retrieve(broader_q, k=4, method="mmr")
            context = self.retriever.format_context(docs)
        
        steps.append(f"Retrieved {len(docs)} docs (relevance: {relevance_score:.2f})")
        
        # Step 5: Build enhanced prompt
        if verbose: print("\n🤖 Step 5: Generating answer...")
        
        tool_context = f"\nTool result: {tool_result}\n" if tool_name else ""
        
        prompt = f"""You are an intelligent assistant. Answer the question based on the context provided.
Think step by step. Only use information from the context. If unsure, say so.

Context:
{context}
{tool_context}
Question: {question}

Answer:"""
        
        answer = self.llm.generate(prompt)
        
        # Save to conversation
        self.conversation.append({"question": question, "answer": answer})
        
        sources = list(set(d.metadata.get('source', '?') for d in docs))
        
        result = {
            "question": question,
            "rewritten_question": rewritten_q,
            "answer": answer,
            "sources": sources,
            "relevance_score": relevance_score,
            "steps": steps
        }
        
        if verbose:
            print(f"\n{'─'*65}")
            print(f"✅ Answer:\n{answer}")
            print(f"\n📚 Sources: {sources}")
            print(f"\n🔎 Steps: {steps}")
        
        return result


# ─── Test the Agentic RAG ─────────────────────────────────────────────────────
agentic_rag = AgenticRAG(vector_store, llm)

print("\n" + "="*65)
print("Testing Agentic RAG with various question types")
print("="*65)

test_questions = [
    "What embedding model produces 384-dimensional vectors?",
    "Explain the difference between FAISS index types",
    "What is the current date?",  # Will trigger tool
]

for q in test_questions:
    agentic_rag.query(q, verbose=True)
    print()


Testing Agentic RAG with various question types

🧠 AGENTIC RAG
💬 Question: What embedding model produces 384-dimensional vectors?

📝 Step 1: Query analysis...
   No rewriting needed

📥 Step 2: Retrieval...
⚙️  Encoding 1 texts...
⚙️  Encoding 1 texts...
⚙️  Encoding 8 texts...
   Relevance score: 0.40

🤖 Step 5: Generating answer...

─────────────────────────────────────────────────────────────────
✅ Answer:
[MOCK ANSWER] Based on the provided context, I found relevant information.
Question asked: 'What embedding model produces 384-dimensional vectors?

Answer:'
Context excerpt: '[Source 1: vector_db_guide.txt]
oding text with a model like sentence-transformers All-MiniLM-L6-v2 is a popular free model that creates 384-dimension...'
(Replace MockLLM with a real model for actual answers)

📚 Sources: ['ai_basics.txt', 'python_guide.txt', 'vector_db_guide.txt']

🔎 Steps: ['Query: What embedding model produces 384-dimensional vectors?', 'Retrieved 4 docs (relevance: 0.40)']


🧠 AGENTIC RAG

## 10. Evaluation & Testing <a id='evaluation'></a>

### Theory: RAG Evaluation Metrics

| Metric | What it measures | How |
|--------|-----------------|-----|
| **Faithfulness** | Is answer supported by context? | Check if claims appear in retrieved docs |
| **Answer Relevance** | Does answer address the question? | Embed answer, query; measure similarity |
| **Context Relevance** | Were retrieved docs relevant? | Measure doc-query similarity |
| **Context Recall** | Were all relevant docs retrieved? | Compare to ground truth |
| **Answer Correctness** | Is the answer factually correct? | Compare to reference answer |

In [11]:
class RAGEvaluator:
    """
    Simple RAG evaluation framework.
    Uses embedding similarity for metric approximation.
    """
    
    def __init__(self, embedding_model: SentenceTransformer):
        self.model = embedding_model
    
    def answer_relevance(self, question: str, answer: str) -> float:
        """How well does the answer address the question? (0-1)"""
        if not answer.strip():
            return 0.0
        embs = self.model.encode([question, answer], normalize_embeddings=True)
        return float(np.dot(embs[0], embs[1]))
    
    def context_relevance(self, question: str, docs: List[Document]) -> float:
        """How relevant are the retrieved docs to the question? (0-1)"""
        if not docs:
            return 0.0
        q_emb = self.model.encode([question], normalize_embeddings=True)[0]
        doc_embs = self.model.encode(
            [d.content for d in docs], normalize_embeddings=True
        )
        scores = np.dot(doc_embs, q_emb)
        return float(np.mean(scores))
    
    def faithfulness(self, answer: str, docs: List[Document]) -> float:
        """Is the answer grounded in the retrieved context? (0-1)"""
        if not docs or not answer.strip():
            return 0.0
        # Split answer into sentences
        sentences = [s.strip() for s in re.split(r'[.!?]', answer) if len(s.strip()) > 10]
        if not sentences:
            return 0.5
        
        context_text = " ".join([d.content for d in docs])
        all_texts = sentences + [context_text]
        embs = self.model.encode(all_texts, normalize_embeddings=True)
        
        sentence_embs = embs[:len(sentences)]
        context_emb = embs[-1]
        
        scores = np.dot(sentence_embs, context_emb)
        return float(np.mean(scores))
    
    def evaluate(
        self, 
        question: str, 
        answer: str, 
        docs: List[Document],
        reference_answer: str = None
    ) -> Dict:
        """Run full evaluation suite."""
        metrics = {
            "answer_relevance": self.answer_relevance(question, answer),
            "context_relevance": self.context_relevance(question, docs),
            "faithfulness": self.faithfulness(answer, docs),
        }
        
        if reference_answer:
            ans_embs = self.model.encode(
                [answer, reference_answer], normalize_embeddings=True
            )
            metrics["answer_similarity"] = float(np.dot(ans_embs[0], ans_embs[1]))
        
        # Overall RAG score
        core_metrics = [metrics["answer_relevance"], 
                       metrics["context_relevance"],
                       metrics["faithfulness"]]
        metrics["overall_score"] = float(np.mean(core_metrics))
        
        return metrics


# ─── Run evaluation ───────────────────────────────────────────────────────────
evaluator = RAGEvaluator(vector_store.model)

# Evaluation test set
test_set = [
    {
        "question": "What is FAISS used for?",
        "reference": "FAISS is used for efficient similarity search on high-dimensional vectors."
    },
    {
        "question": "What libraries does Python have for data science?",
        "reference": "Python has NumPy for numerical computing, Pandas for data manipulation, and Matplotlib for visualization."
    },
    {
        "question": "What is the Transformer architecture?",
        "reference": "Transformers are neural network architectures using self-attention mechanisms, introduced in 2017."
    }
]

print("📊 RAG Evaluation Results")
print("="*65)

all_scores = []
for test in test_set:
    q = test["question"]
    docs = retriever.retrieve(q, k=3)
    context = retriever.format_context(docs)
    answer = llm.generate(f"Context:\n{context}\n\nQuestion: {q}\n\nAnswer:")
    
    metrics = evaluator.evaluate(q, answer, docs, test["reference"])
    all_scores.append(metrics)
    
    print(f"\n❓ {q}")
    for k, v in metrics.items():
        bar = '█' * int(v * 10) + '░' * (10 - int(v * 10))
        print(f"   {k:25s}: {bar} {v:.3f}")

# Average scores
avg_metrics = {}
for key in all_scores[0].keys():
    avg_metrics[key] = np.mean([s[key] for s in all_scores])

print("\n" + "─"*65)
print("📈 AVERAGE SCORES:")
for k, v in avg_metrics.items():
    bar = '█' * int(v * 10) + '░' * (10 - int(v * 10))
    print(f"   {k:25s}: {bar} {v:.3f}")

📊 RAG Evaluation Results
⚙️  Encoding 1 texts...

❓ What is FAISS used for?
   answer_relevance         : █████░░░░░ 0.581
   context_relevance        : █░░░░░░░░░ 0.190
   faithfulness             : ██░░░░░░░░ 0.263
   answer_similarity        : ███░░░░░░░ 0.365
   overall_score            : ███░░░░░░░ 0.345
⚙️  Encoding 1 texts...

❓ What libraries does Python have for data science?
   answer_relevance         : ██████░░░░ 0.607
   context_relevance        : █████░░░░░ 0.532
   faithfulness             : ██░░░░░░░░ 0.255
   answer_similarity        : ███░░░░░░░ 0.330
   overall_score            : ████░░░░░░ 0.465
⚙️  Encoding 1 texts...

❓ What is the Transformer architecture?
   answer_relevance         : █████░░░░░ 0.542
   context_relevance        : ██░░░░░░░░ 0.251
   faithfulness             : ██░░░░░░░░ 0.299
   answer_similarity        : █████░░░░░ 0.561
   overall_score            : ███░░░░░░░ 0.364

─────────────────────────────────────────────────────────────────
📈 AVERAGE 

## 11. Production Tips <a id='production'></a>

In [12]:
# ─── Tip 1: Caching embeddings ────────────────────────────────────────────────
import hashlib

class CachedEmbedder:
    """Cache embeddings to avoid recomputing."""
    
    def __init__(self, model: SentenceTransformer):
        self.model = model
        self.cache = {}
    
    def embed(self, text: str) -> np.ndarray:
        key = hashlib.md5(text.encode()).hexdigest()
        if key not in self.cache:
            self.cache[key] = self.model.encode([text], normalize_embeddings=True)[0]
        return self.cache[key]
    
    def cache_stats(self):
        return {"cached_items": len(self.cache), 
                "memory_mb": len(self.cache) * 384 * 4 / 1e6}


# ─── Tip 2: Document preprocessing pipeline ───────────────────────────────────
class DocumentPreprocessor:
    """Clean and normalize documents before indexing."""
    
    @staticmethod
    def clean(text: str) -> str:
        text = re.sub(r'\s+', ' ', text)        # Normalize whitespace
        text = re.sub(r'[\x00-\x08\x0b-\x0c\x0e-\x1f]', '', text)  # Control chars
        text = re.sub(r'(.)\1{4,}', r'\1\1\1', text)  # Repeated chars (e.g. -----)
        return text.strip()
    
    @staticmethod
    def load_text_file(filepath: str) -> Document:
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read()
        return Document(
            content=DocumentPreprocessor.clean(content),
            metadata={'source': Path(filepath).name, 'type': 'text'},
            doc_id=Path(filepath).stem
        )
    
    @staticmethod
    def load_pdf(filepath: str) -> List[Document]:
        """Load PDF — requires pypdf."""
        try:
            from pypdf import PdfReader
            reader = PdfReader(filepath)
            docs = []
            for i, page in enumerate(reader.pages):
                text = page.extract_text()
                if text.strip():
                    docs.append(Document(
                        content=DocumentPreprocessor.clean(text),
                        metadata={'source': Path(filepath).name, 'page': i+1},
                        doc_id=f"{Path(filepath).stem}_p{i+1}"
                    ))
            return docs
        except ImportError:
            print("Install pypdf: pip install pypdf")
            return []


# ─── Tip 3: Production-ready index selection guide ────────────────────────────
def recommend_index_type(n_documents: int) -> str:
    """
    Recommend FAISS index type based on dataset size.
    
    Rules of thumb:
    - < 10k vectors:   IndexFlatL2 (exact, fast enough)
    - 10k – 100k:      IndexIVFFlat (approximate, trained clusters)
    - 100k – 10M:      IndexHNSWFlat (graph-based, no training needed)
    - > 10M:           IndexIVFPQ (compressed, requires training)
    """
    if n_documents < 10_000:
        return "flat", "IndexFlatL2 — exact search, ideal for your size"
    elif n_documents < 100_000:
        return "ivf", "IndexIVFFlat — approximate, good speed/accuracy tradeoff"
    elif n_documents < 10_000_000:
        return "hnsw", "IndexHNSWFlat — graph-based, best for production"
    else:
        return "pq", "IndexIVFPQ — compressed, necessary for this scale"


print("Index Recommendations by Scale:")
print("─" * 50)
for n in [100, 1_000, 50_000, 500_000, 50_000_000]:
    idx_type, reason = recommend_index_type(n)
    print(f"  {n:>12,} docs → {idx_type:8s} | {reason}")

print("\n Production checklist:")
checklist = [
    "Use HNSW index for >10k documents",
    "Save FAISS index to disk with faiss.write_index()",
    "Cache query embeddings to avoid recomputation",
    "Set score threshold to filter irrelevant results",
    "Log queries and answers for continuous improvement",
    "Use MMR retrieval to avoid redundant results",
    "Add document metadata for provenance tracking",
    "Run periodic re-indexing as knowledge base grows",
    "Monitor faithfulness scores to detect degradation",
    "Use larger embedding models (all-mpnet-base-v2) for better quality"
]
for item in checklist:
    print(f"  ✓ {item}")

📋 Index Recommendations by Scale:
──────────────────────────────────────────────────
           100 docs → flat     | IndexFlatL2 — exact search, ideal for your size
         1,000 docs → flat     | IndexFlatL2 — exact search, ideal for your size
        50,000 docs → ivf      | IndexIVFFlat — approximate, good speed/accuracy tradeoff
       500,000 docs → hnsw     | IndexHNSWFlat — graph-based, best for production
    50,000,000 docs → pq       | IndexIVFPQ — compressed, necessary for this scale

✅ Production checklist:
  ✓ Use HNSW index for >10k documents
  ✓ Save FAISS index to disk with faiss.write_index()
  ✓ Cache query embeddings to avoid recomputation
  ✓ Set score threshold to filter irrelevant results
  ✓ Log queries and answers for continuous improvement
  ✓ Use MMR retrieval to avoid redundant results
  ✓ Add document metadata for provenance tracking
  ✓ Run periodic re-indexing as knowledge base grows
  ✓ Monitor faithfulness scores to detect degradation
  ✓ Use larger em

In [13]:
# ─── Full Pipeline Summary ────────────────────────────────────────────────────

print("""
╔══════════════════════════════════════════════════════════════╗
║          RAG AGENT — COMPLETE PIPELINE SUMMARY               ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  INDEXING (one-time):                                        ║
║  1. Load documents (PDF, TXT, DOCX, web)                     ║
║  2. Clean & preprocess text                                  ║
║  3. Split into chunks (TextSplitter, chunk_size=500)         ║
║  4. Embed chunks (SentenceTransformer)                       ║
║  5. Build FAISS index (choose type by scale)                 ║
║  6. Save index + documents to disk                           ║
║                                                              ║
║  QUERYING (per question):                                    ║
║  1. Receive user question                                    ║
║  2. (Optional) Rewrite query using history                   ║
║  3. Embed question (same model as indexing!)                 ║
║  4. FAISS similarity search → top-K docs                     ║
║  5. Grade relevance, retry if needed                         ║
║  6. Build prompt: System + Context + Question                ║
║  7. LLM generates answer                                     ║
║  8. Return answer + sources + metadata                       ║
║                                                              ║
║  FREE STACK:                                                 ║
║  Embeddings : sentence-transformers (all-MiniLM-L6-v2)      ║
║  Vector DB  : FAISS (faiss-cpu)                              ║
║  LLM (local): HuggingFace (flan-t5, TinyLlama) or Ollama    ║
║  LLM (cloud): Groq API free tier (Llama3 70B!)              ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════╗
║          RAG AGENT — COMPLETE PIPELINE SUMMARY               ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  INDEXING (one-time):                                        ║
║  1. Load documents (PDF, TXT, DOCX, web)                     ║
║  2. Clean & preprocess text                                  ║
║  3. Split into chunks (TextSplitter, chunk_size=500)         ║
║  4. Embed chunks (SentenceTransformer)                       ║
║  5. Build FAISS index (choose type by scale)                 ║
║  6. Save index + documents to disk                           ║
║                                                              ║
║  QUERYING (per question):                                    ║
║  1. Receive user question                                    ║
║  2. (Optional) Rewrite query using history                   ║
║  3. Embed question (sa

---
## Key Takeaways

| Concept | What to Remember |
|---------|------------------|
| **RAG** | Retrieval + Generation — grounds LLMs in real data |
| **FAISS** | Free, fast vector search — use `flat` for small, `HNSW` for large |
| **Embeddings** | `all-MiniLM-L6-v2` is the best free embedding model for most tasks |
| **Chunking** | 400–600 tokens with 10% overlap is the sweet spot |
| **MMR** | Always use MMR retrieval over pure similarity to avoid redundancy |
| **Free LLMs** | Groq's free tier (Llama 70B) or Ollama locally give great results |
| **Evaluation** | Track faithfulness + answer_relevance + context_relevance |

## Further Resources

- **FAISS docs**: https://github.com/facebookresearch/faiss/wiki
- **Sentence Transformers**: https://www.sbert.net
- **Groq free API**: https://console.groq.com
- **Ollama**: https://ollama.ai
- **LangChain RAG**: https://python.langchain.com/docs/use_cases/question_answering
- **RAGAS (eval framework)**: https://github.com/explodinggradients/ragas

---
*Built from scratch — no LangChain dependencies, fully understandable, production-ready.*